# DMS-Validation Correlation Analysis

This notebook correlates DMS escape scores with IC50 values from validation strains containing those point mutations.

**Goal**: Validate that DMS predictions match tradiational neutralization assays.

**Approach**:
- Load validation strain mutations from `validation_mutations.xlsx`
- Load IC50 data from neutralization experiments
- Load DMS escape scores
- Merge datasets by mutation
- Create scatter plots: DMS score (x-axis, linear) vs IC50 (y-axis, log scale)
- Calculate correlation statistics

**Notes**
- B1 T67N and Q209K mutants are B1 background with point mutations TO the Long residue (so WT DMS residue) at that site and are NOT mAb DMS escape mutations, and are therefore NOT included in the correlation plots. However, the mutations in Long N67T and K209Q ARE mAb DMS escape mutations because they are mutations away from the DMS strain and therefore ARE included in the correlation plots. 
- Known nirsevimab escape mutations in RSV-B include 201S/T and 68Q/K. These could also be included in the correlation analysis. 
- Experiments were run on different days, so two sets of plots are generated one using measured IC50s and for the WT the geometric mean from all experimental days and one using fold change IC50 from WT (ran on the same day). To account for any variation in antibody dilution, we will use fold change IC50 from WT for the final correlation analysis. 

In [1]:
# Imports
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import altair as alt
from scipy import stats
from pathlib import Path

# Set working directory to project root
# This ensures the path is always correct regardless of kernel state
repo_root = Path('/fh/fast/bloom_j/computational_notebooks/csimonich/2025/RSV_Long_F_DMS/RSV_Long_F_DMS/non-pipeline_analyses/validations/')
os.chdir(repo_root)
print(f"Working directory: {os.getcwd()}")


# Allow more rows for Altair
alt.data_transformers.disable_max_rows()

print("✓ Imports and setup complete")



Working directory: /fh/fast/bloom_j/computational_notebooks/csimonich/2025/RSV_Long_F_DMS/RSV_Long_F_DMS/non-pipeline_analyses/validations
✓ Imports and setup complete


In [2]:
# Helpers for legend ordering by site
import re

def mutation_sort_key_wt_first(label):
    label_str = str(label)
    if label_str.startswith('WT '):
        return (0, 0, label_str)
    match = re.search(r'(\d+)([A-Z])$', label_str)
    if match:
        return (1, int(match.group(1)), label_str)
    return (2, 10**9, label_str)

def mutation_sort_key_site_only(label):
    label_str = str(label)
    match = re.search(r'(\d+)([A-Z])$', label_str)
    if match:
        return (0, int(match.group(1)), label_str)
    return (1, 10**9, label_str)



In [3]:
import altair as alt
import numpy as np
import pandas as pd
from scipy import stats
import scipy.stats as scipy_stats
import re

alt.data_transformers.disable_max_rows()

# --- Helpers for legend ordering and tight domains ---

def mutation_sort_key_wt_first(label):
    label_str = str(label)
    if label_str.startswith('WT '):
        return (0, 0, label_str)
    match = re.search(r'(\d+)([A-Z])$', label_str)
    if match:
        return (1, int(match.group(1)), label_str)
    return (2, 10**9, label_str)


def _tight_linear_domain(values, pad_fraction=0.05, fallback_pad=0.01):
    vmin = float(np.nanmin(values))
    vmax = float(np.nanmax(values))
    pad = (vmax - vmin) * pad_fraction
    if pad == 0:
        pad = max(abs(vmin) * pad_fraction, fallback_pad)
    return [vmin - pad, vmax + pad]


def _tight_log_domain(values, low_pad=0.9, high_pad=1.1):
    values = np.array(values)
    values = values[values > 0]
    if len(values) == 0:
        return [1e-4, 1e4]
    vmin = float(np.nanmin(values))
    vmax = float(np.nanmax(values))
    return [vmin * low_pad, vmax * high_pad]


def create_individual_plot(data, strain_bg, antibody, fmt, global_color_mapping):
    subset = data[
        (data['strain_background'] == strain_bg) &
        (data['antibody'] == antibody) &
        (data['format'] == fmt)
    ].copy()

    if len(subset) < 2:
        return None, None

    if subset['is_wt'].sum() > 0:
        wt_rows = subset[subset['is_wt']].copy()
        geometric_mean_ic50 = np.exp(np.log(wt_rows['ic50']).mean())
        wt_single_row = wt_rows.iloc[0].copy()
        wt_single_row['ic50'] = geometric_mean_ic50
        wt_single_row['ic50_str'] = f"{geometric_mean_ic50:.4g}"
        wt_single_row['strain'] = f"WT {strain_bg} (geomean)"
        subset = subset[~subset['is_wt']].copy()
        subset = pd.concat([subset, pd.DataFrame([wt_single_row])], ignore_index=True)

    def make_mutation_label(row):
        if row.get('is_wt', False):
            return f"WT {strain_bg} (geomean)"
        if pd.isna(row.get('site')) or pd.isna(row.get('mutant')):
            return "Unmutated"
        site = int(row['site'])
        wt = row.get('wt_residue_strain', row.get('wt_residue_dms', ''))
        return f"{wt}{site}{row.get('mutant','')}"

    subset['mutation_label'] = subset.apply(make_mutation_label, axis=1)
    mutations_in_plot = sorted(subset['mutation_label'].unique(), key=mutation_sort_key_wt_first)

    color_scale = alt.Scale(
        domain=mutations_in_plot,
        range=[('#000000' if m in ['WT Long (geomean)', 'WT B1 (geomean)'] else global_color_mapping.get(m, '#999999')) for m in mutations_in_plot]
    )

    legend = alt.Legend(
        labelExpr="indexof(datum.label, 'WT ') == 0 ? 'wildtype' : datum.label"
    )

    subset_measured = subset[~subset['at_upper_bound']]
    subset_bound = subset[subset['at_upper_bound']]

    n_at_bound = int(subset['at_upper_bound'].sum())
    n_measured = len(subset) - n_at_bound

    x_domain = _tight_linear_domain(subset['escape'])
    y_domain = _tight_log_domain(subset['ic50'])

    x_scale = alt.Scale(zero=False, domain=x_domain)
    y_scale = alt.Scale(type='log', domain=y_domain)

    points_measured = alt.Chart(subset_measured).mark_circle(
        size=300, opacity=0.85, stroke='black', strokeWidth=0.5
    ).encode(
        x=alt.X('escape:Q', title='Effect measured by deep mutational scanning', scale=x_scale),
        y=alt.Y('ic50:Q', title='Pseudovirus neutralization IC50 (nM)', scale=y_scale),
        color=alt.Color('mutation_label:N', title='Mutation', scale=color_scale, sort=mutations_in_plot, legend=legend),
        tooltip=[
            alt.Tooltip('strain:N', title='Strain'),
            alt.Tooltip('mutation_label:N', title='Mutation'),
            alt.Tooltip('escape:Q', title='DMS Escape', format='.3f'),
            alt.Tooltip('ic50_str:N', title='IC50')
        ]
    )

    points_bound = alt.Chart(subset_bound).mark_circle(
        size=350,
        opacity=0.95,
        stroke='black',
        strokeWidth=0.5
    ).encode(
        x=alt.X('escape:Q', scale=x_scale),
        y=alt.Y('ic50:Q', scale=y_scale),
        color=alt.Color('mutation_label:N', title='Mutation', scale=color_scale, sort=mutations_in_plot, legend=legend),
        tooltip=[
            alt.Tooltip('strain:N', title='Strain'),
            alt.Tooltip('mutation_label:N', title='Mutation'),
            alt.Tooltip('escape:Q', title='DMS Escape', format='.3f'),
            alt.Tooltip('ic50_str:N', title='IC50 (upper limit)')
        ]
    )

    layers = [points_measured, points_bound]

    escape_unique = subset['escape'].nunique(dropna=True)
    pearson_r = pearson_p = np.nan
    if escape_unique >= 2:
        pearson_r, pearson_p = scipy_stats.pearsonr(
            subset['escape'], np.log10(subset['ic50'])
        )
        z = np.polyfit(subset['escape'], np.log10(subset['ic50']), 1)
        p = np.poly1d(z)
        xs = np.linspace(subset['escape'].min(), subset['escape'].max(), 200)
        ys = 10 ** p(xs)
        reg_df = pd.DataFrame({'escape': xs, 'ic50': ys})
        reg_line = alt.Chart(reg_df).mark_line(
            strokeDash=[6, 4], strokeWidth=0.5, opacity=0.7, color='black'
        ).encode(
            x=alt.X('escape:Q', scale=x_scale),
            y=alt.Y('ic50:Q', scale=y_scale)
        )
        layers.append(reg_line)

    if n_at_bound > 0:
        lod_val = subset_bound['ic50'].max()
        lod_df = pd.DataFrame({'ic50': [lod_val]})
        lod_rule = alt.Chart(lod_df).mark_rule(
            strokeDash=[2, 3], strokeWidth=0.5, opacity=0.6, color='black'
        ).encode(
            y=alt.Y('ic50:Q', scale=y_scale)
        )
        layers.append(lod_rule)

    r_label = "r = NA" if not np.isfinite(pearson_r) else f"r = {pearson_r:.2f}"
    r_df = pd.DataFrame({
        'escape': [subset['escape'].min()],
        'ic50': [subset['ic50'].max() * 0.9],
        'label': [r_label]
    })
    r_text = alt.Chart(r_df).mark_text(
        align='left', baseline='top', dx=6, dy=6, fontSize=24
    ).encode(
        x=alt.X('escape:Q', scale=x_scale),
        y=alt.Y('ic50:Q', scale=y_scale),
        text='label:N'
    )
    layers.append(r_text)

    chart = alt.layer(*layers).properties(
        width=350,
        height=280,
        title=f"{antibody} {fmt} — {strain_bg}"
    ).configure_view(
        strokeWidth=0
    ).configure_axis(
        labelFontSize=24,
        titleFontSize=26,
        labelFont='Arial',
        titleFont='Arial',
        grid=False,
        domainWidth=0.5,
        tickWidth=0.5,
        domainColor='black',
        tickColor='black',
        tickSize=5,
        tickCount=5
    ).configure_legend(
        titleFontSize=24,
        labelFontSize=22,
        labelFont='Arial',
        titleFont='Arial',
        strokeWidth=0.5,
        symbolStrokeWidth=0.5,
        symbolSize=250
    ).configure_title(
        fontSize=26,
        font='Arial'
    ).configure_text(
        font='Arial'
    )

    stats_dict = {
        'strain_background': strain_bg,
        'antibody': antibody,
        'format': fmt,
        'n_total': len(subset),
        'n_measured': n_measured,
        'n_at_bound': n_at_bound,
        'pearson_r': pearson_r,
        'pearson_p': pearson_p
    }

    return chart, stats_dict


def create_fold_change_plot(data, strain_bg, antibody, fmt, global_color_mapping):
    subset = data[
        (data['strain_background'] == strain_bg) &
        (data['antibody'] == antibody) &
        (data['format'] == fmt)
    ].copy()

    if len(subset) < 2:
        return None, None

    if subset['is_wt'].sum() > 0:
        wt_rows = subset[subset['is_wt']].copy()
        geometric_mean_ic50 = np.exp(np.log(wt_rows['ic50']).mean())
        wt_single_row = wt_rows.iloc[0].copy()
        wt_single_row['ic50'] = geometric_mean_ic50
        wt_single_row['ic50_str'] = f"{geometric_mean_ic50:.4g}"
        wt_single_row['fold_change'] = 1.0
        wt_single_row['strain'] = f"WT {strain_bg} (geomean)"
        subset = subset[~subset['is_wt']].copy()
        subset = pd.concat([subset, pd.DataFrame([wt_single_row])], ignore_index=True)

    def make_mutation_label(row):
        if row.get('is_wt', False):
            return f"WT {strain_bg} (geomean)"
        if pd.isna(row.get('site')) or pd.isna(row.get('mutant')):
            return "Unmutated"
        site = int(row['site'])
        wt = row.get('wt_residue_strain', row.get('wt_residue_dms', ''))
        return f"{wt}{site}{row.get('mutant','')}"

    subset['mutation_label'] = subset.apply(make_mutation_label, axis=1)
    mutations_in_plot = sorted(subset['mutation_label'].unique(), key=mutation_sort_key_wt_first)

    color_scale = alt.Scale(
        domain=mutations_in_plot,
        range=[('#000000' if m in ['WT Long (geomean)', 'WT B1 (geomean)'] else global_color_mapping.get(m, '#999999')) for m in mutations_in_plot]
    )

    legend = alt.Legend(
        labelExpr="indexof(datum.label, 'WT ') == 0 ? 'wildtype' : datum.label"
    )

    subset_measured = subset[~subset['at_upper_bound']]
    subset_bound = subset[subset['at_upper_bound']]

    x_domain = _tight_linear_domain(subset['escape'])
    y_domain = _tight_log_domain(subset['fold_change'])

    x_scale = alt.Scale(zero=False, domain=x_domain)
    y_scale = alt.Scale(type='log', domain=y_domain)

    points_measured = alt.Chart(subset_measured).mark_circle(
        size=300, opacity=0.85, stroke='black', strokeWidth=0.5
    ).encode(
        x=alt.X('escape:Q', title='Effect measured by deep mutational scanning', scale=x_scale),
        y=alt.Y('fold_change:Q', title='Fold change IC50 from WT', scale=y_scale),
        color=alt.Color('mutation_label:N', title='Mutation', scale=color_scale, sort=mutations_in_plot, legend=legend),
        tooltip=[
            alt.Tooltip('strain:N', title='Strain'),
            alt.Tooltip('mutation_label:N', title='Mutation'),
            alt.Tooltip('escape:Q', title='DMS Escape', format='.3f'),
            alt.Tooltip('fold_change:Q', title='Fold change', format='.3f')
        ]
    )

    points_bound = alt.Chart(subset_bound).mark_circle(
        size=350, opacity=0.95, stroke='black', strokeWidth=0.5
    ).encode(
        x=alt.X('escape:Q', scale=x_scale),
        y=alt.Y('fold_change:Q', scale=y_scale),
        color=alt.Color('mutation_label:N', title='Mutation', scale=color_scale, sort=mutations_in_plot, legend=legend),
        tooltip=[
            alt.Tooltip('strain:N', title='Strain'),
            alt.Tooltip('mutation_label:N', title='Mutation'),
            alt.Tooltip('escape:Q', title='DMS Escape', format='.3f'),
            alt.Tooltip('fold_change:Q', title='Fold change (upper limit)', format='.3f')
        ]
    )

    layers = [points_measured, points_bound]

    escape_unique = subset['escape'].nunique(dropna=True)
    pearson_r = pearson_p = np.nan
    if escape_unique >= 2:
        pearson_r, pearson_p = scipy_stats.pearsonr(
            subset['escape'], np.log10(subset['fold_change'])
        )
        z = np.polyfit(subset['escape'], np.log10(subset['fold_change']), 1)
        p = np.poly1d(z)
        xs = np.linspace(subset['escape'].min(), subset['escape'].max(), 200)
        ys = 10 ** p(xs)
        reg_df = pd.DataFrame({'escape': xs, 'fold_change': ys})
        reg_line = alt.Chart(reg_df).mark_line(
            strokeDash=[6, 4], strokeWidth=0.5, opacity=0.7, color='black'
        ).encode(
            x=alt.X('escape:Q', scale=x_scale),
            y=alt.Y('fold_change:Q', scale=y_scale)
        )
        layers.append(reg_line)

    if len(subset_bound) > 0:
        lod_val = subset_bound['fold_change'].max()
        lod_df = pd.DataFrame({'fold_change': [lod_val]})
        lod_rule = alt.Chart(lod_df).mark_rule(
            strokeDash=[2, 3], strokeWidth=0.5, opacity=0.6, color='black'
        ).encode(
            y=alt.Y('fold_change:Q', scale=y_scale)
        )
        layers.append(lod_rule)

    r_label = "r = NA" if not np.isfinite(pearson_r) else f"r = {pearson_r:.2f}"
    r_df = pd.DataFrame({
        'escape': [subset['escape'].min()],
        'fold_change': [subset['fold_change'].max() * 0.9],
        'label': [r_label]
    })
    r_text = alt.Chart(r_df).mark_text(
        align='left', baseline='top', dx=6, dy=6, fontSize=24
    ).encode(
        x=alt.X('escape:Q', scale=x_scale),
        y=alt.Y('fold_change:Q', scale=y_scale),
        text='label:N'
    )
    layers.append(r_text)

    chart = alt.layer(*layers).properties(
        width=350,
        height=280,
        title=f"{antibody} {fmt} — {strain_bg}"
    ).configure_view(
        strokeWidth=0
    ).configure_axis(
        labelFontSize=24,
        titleFontSize=26,
        labelFont='Arial',
        titleFont='Arial',
        grid=False,
        domainWidth=0.5,
        tickWidth=0.5,
        domainColor='black',
        tickColor='black',
        tickSize=5,
        tickCount=5
    ).configure_legend(
        titleFontSize=24,
        labelFontSize=22,
        labelFont='Arial',
        titleFont='Arial',
        strokeWidth=0.5,
        symbolStrokeWidth=0.5,
        symbolSize=250
    ).configure_title(
        fontSize=26,
        font='Arial'
    ).configure_text(
        font='Arial'
    )

    stats_dict = {
        'strain_background': strain_bg,
        'antibody': antibody,
        'format': fmt,
        'n_total': len(subset),
        'pearson_r': pearson_r,
        'pearson_p': pearson_p
    }

    return chart, stats_dict


def create_format_comparison_plot(data, strain_bg, antibody, global_color_mapping):
    subset = data[
        (data['strain_background'] == strain_bg) &
        (data['antibody'] == antibody)
    ].copy()

    if len(subset) < 2:
        return None, None

    pivot_data = subset.pivot_table(
        index=['strain', 'site', 'mutant', 'wt_residue_strain', 'is_wt', 'strain_background'],
        columns='format',
        values='fold_change',
        aggfunc='first'
    ).reset_index()

    pivot_bound = subset.pivot_table(
        index=['strain', 'site', 'mutant', 'wt_residue_strain', 'is_wt', 'strain_background'],
        columns='format',
        values='at_upper_bound',
        aggfunc='max'
    ).reset_index()

    pivot_data = pivot_data.dropna(subset=['Fab', 'IgG'])
    if len(pivot_data) < 2:
        return None, None

    pivot_data = pivot_data[
        (pivot_data['Fab'] > 0) &
        (pivot_data['IgG'] > 0)
    ].copy()
    if len(pivot_data) < 2:
        return None, None

    pivot_data = pivot_data.merge(
        pivot_bound,
        on=['strain', 'site', 'mutant', 'wt_residue_strain', 'is_wt', 'strain_background'],
        how='left',
        suffixes=('', '_bound')
    )
    pivot_data['at_upper_fab'] = pivot_data.get('Fab_bound', False).fillna(False)
    pivot_data['at_upper_igg'] = pivot_data.get('IgG_bound', False).fillna(False)

    if not pivot_data['is_wt'].any():
        pivot_data = pd.concat([
            pivot_data,
            pd.DataFrame([{
                'strain': f"WT {strain_bg} (geomean)",
                'site': None,
                'mutant': None,
                'wt_residue_strain': None,
                'is_wt': True,
                'strain_background': strain_bg,
                'Fab': 1.0,
                'IgG': 1.0,
                'Fab_bound': False,
                'IgG_bound': False,
                'at_upper_fab': False,
                'at_upper_igg': False
            }])
        ], ignore_index=True)

    def make_mutation_label(row):
        if row.get('is_wt', False):
            return f"WT {strain_bg} (geomean)"
        if pd.isna(row.get('site')) or pd.isna(row.get('mutant')):
            return "Unmutated"
        site = int(row['site'])
        wt = row.get('wt_residue_strain', row.get('wt_residue_dms', ''))
        return f"{wt}{site}{row.get('mutant','')}"

    pivot_data['mutation_label'] = pivot_data.apply(make_mutation_label, axis=1)
    mutations_in_plot = sorted(pivot_data['mutation_label'].unique(), key=mutation_sort_key_wt_first)

    color_scale = alt.Scale(
        domain=mutations_in_plot,
        range=[('#000000' if m in ['WT Long (geomean)', 'WT B1 (geomean)'] else global_color_mapping.get(m, '#999999')) for m in mutations_in_plot]
    )

    legend = alt.Legend(
        labelExpr="indexof(datum.label, 'WT ') == 0 ? 'wildtype' : datum.label"
    )

    min_fc = min(pivot_data['Fab'].min(), pivot_data['IgG'].min())
    max_fc = max(pivot_data['Fab'].max(), pivot_data['IgG'].max())
    domain = _tight_log_domain([min_fc, max_fc])

    x_scale = alt.Scale(type='log', domain=domain)
    y_scale = alt.Scale(type='log', domain=domain)

    points = alt.Chart(pivot_data).mark_circle(
        size=300, opacity=0.85, stroke='black', strokeWidth=0.5
    ).encode(
        x=alt.X('Fab:Q', title='Fab fold-change vs WT', scale=x_scale),
        y=alt.Y('IgG:Q', title='IgG fold-change vs WT', scale=y_scale),
        color=alt.Color('mutation_label:N', title='Mutation', scale=color_scale, sort=mutations_in_plot, legend=legend),
        tooltip=[
            alt.Tooltip('mutation_label:N', title='Mutation'),
            alt.Tooltip('Fab:Q', title='Fab fold-change', format='.3f'),
            alt.Tooltip('IgG:Q', title='IgG fold-change', format='.3f')
        ]
    )

    layers = [points]

    min_fc = min(pivot_data['Fab'].min(), pivot_data['IgG'].min())
    max_fc = max(pivot_data['Fab'].max(), pivot_data['IgG'].max())
    xs = np.logspace(np.log10(min_fc), np.log10(max_fc), 200)
    id_df = pd.DataFrame({'Fab': xs, 'IgG': xs})
    identity_line = alt.Chart(id_df).mark_line(
        strokeDash=[4, 4], strokeWidth=0.75, opacity=0.6, color='#808080'
    ).encode(
        x=alt.X('Fab:Q', scale=x_scale),
        y=alt.Y('IgG:Q', scale=y_scale)
    )
    layers.append(identity_line)

    if pivot_data['at_upper_fab'].any():
        lod_fab = pivot_data.loc[pivot_data['at_upper_fab'], 'Fab'].max()
        lod_df = pd.DataFrame({'Fab': [lod_fab]})
        lod_rule = alt.Chart(lod_df).mark_rule(
            strokeDash=[2, 3], strokeWidth=0.5, opacity=0.6, color='black'
        ).encode(
            x=alt.X('Fab:Q', scale=x_scale)
        )
        layers.append(lod_rule)

    if pivot_data['at_upper_igg'].any():
        lod_igg = pivot_data.loc[pivot_data['at_upper_igg'], 'IgG'].max()
        lod_df = pd.DataFrame({'IgG': [lod_igg]})
        lod_rule = alt.Chart(lod_df).mark_rule(
            strokeDash=[2, 3], strokeWidth=0.5, opacity=0.6, color='black'
        ).encode(
            y=alt.Y('IgG:Q', scale=y_scale)
        )
        layers.append(lod_rule)

    pearson_r = pearson_p = np.nan
    if pivot_data['Fab'].nunique() >= 2 and pivot_data['IgG'].nunique() >= 2:
        pearson_r, pearson_p = scipy_stats.pearsonr(
            np.log10(pivot_data['Fab']), np.log10(pivot_data['IgG'])
        )

    r_label = "r = NA" if not np.isfinite(pearson_r) else f"r = {pearson_r:.2f}"
    r_df = pd.DataFrame({
        'Fab': [pivot_data['Fab'].min()],
        'IgG': [pivot_data['IgG'].max() * 0.9],
        'label': [r_label]
    })
    r_text = alt.Chart(r_df).mark_text(
        align='left', baseline='top', dx=6, dy=6, fontSize=24
    ).encode(
        x=alt.X('Fab:Q', scale=x_scale),
        y=alt.Y('IgG:Q', scale=y_scale),
        text='label:N'
    )
    layers.append(r_text)

    chart = alt.layer(*layers).properties(
        width=350,
        height=280,
        title=f"{antibody} — {strain_bg} (IgG vs Fab)"
    ).configure_view(
        strokeWidth=0
    ).configure_axis(
        labelFontSize=24,
        titleFontSize=26,
        labelFont='Arial',
        titleFont='Arial',
        grid=False,
        domainWidth=0.5,
        tickWidth=0.5,
        domainColor='black',
        tickColor='black',
        tickSize=5,
        tickCount=5
    ).configure_legend(
        titleFontSize=24,
        labelFontSize=22,
        labelFont='Arial',
        titleFont='Arial',
        strokeWidth=0.5,
        symbolStrokeWidth=0.5,
        symbolSize=250
    ).configure_title(
        fontSize=26,
        font='Arial'
    ).configure_text(
        font='Arial'
    )

    stats_dict = {
        'strain_background': strain_bg,
        'antibody': antibody,
        'n_total': len(pivot_data),
        'pearson_r': pearson_r,
        'pearson_p': pearson_p
    }

    return chart, stats_dict


def create_b1_long_comparison_plot(data, antibody, fmt, global_color_mapping):
    subset = data[
        (data['antibody'] == antibody) &
        (data['format'] == fmt) &
        (~data['is_wt'])
    ].copy()

    if len(subset) < 2:
        return None, None

    b1_data = subset[subset['strain_background'] == 'B1'].copy()
    long_data = subset[subset['strain_background'] == 'Long'].copy()

    if len(b1_data) < 1 or len(long_data) < 1:
        return None, None

    b1_merge = b1_data[['site', 'mutant', 'fold_change', 'strain', 'wt_residue_strain', 'at_upper_bound']].copy()
    b1_merge = b1_merge.rename(columns={
        'fold_change': 'fc_b1',
        'strain': 'strain_b1',
        'wt_residue_strain': 'wt_residue_b1',
        'at_upper_bound': 'at_upper_b1'
    })

    long_merge = long_data[['site', 'mutant', 'fold_change', 'strain', 'wt_residue_strain', 'at_upper_bound']].copy()
    long_merge = long_merge.rename(columns={
        'fold_change': 'fc_long',
        'strain': 'strain_long',
        'wt_residue_strain': 'wt_residue_long',
        'at_upper_bound': 'at_upper_long'
    })

    merged = pd.merge(
        b1_merge,
        long_merge,
        on=['site', 'mutant'],
        how='inner'
    )

    if len(merged) < 2:
        return None, None

    # Add WT point at (1,1) if missing
    if 'WT Long (geomean)' not in merged.get('mutation_label', []):
        merged = pd.concat([merged, pd.DataFrame([{
            'mutation_label': 'WT Long (geomean)',
            'fc_b1': 1.0,
            'fc_long': 1.0
        }])], ignore_index=True)


    # Ensure at_upper flags are boolean
    if 'at_upper_b1' in merged.columns:
        merged['at_upper_b1'] = merged['at_upper_b1'].fillna(False)
    else:
        merged['at_upper_b1'] = False
    if 'at_upper_long' in merged.columns:
        merged['at_upper_long'] = merged['at_upper_long'].fillna(False)
    else:
        merged['at_upper_long'] = False

    if 'WT Long (geomean)' not in merged.get('mutation_label', []):
        merged = pd.concat([merged, pd.DataFrame([{
            'mutation_label': 'WT Long (geomean)',
            'fc_b1': 1.0,
            'fc_long': 1.0
        }])], ignore_index=True)


    # Ensure at_upper flags are boolean
    if 'at_upper_b1' in merged.columns:
        merged['at_upper_b1'] = merged['at_upper_b1'].fillna(False)
    else:
        merged['at_upper_b1'] = False
    if 'at_upper_long' in merged.columns:
        merged['at_upper_long'] = merged['at_upper_long'].fillna(False)
    else:
        merged['at_upper_long'] = False

    merged['mutation_label'] = merged.apply(
        lambda r: r.get('mutation_label') if str(r.get('mutation_label')) == 'WT Long (geomean)' else f"{r['wt_residue_long']}{int(r['site'])}{r['mutant']}",
        axis=1
    )
    mutations_in_plot = sorted(merged['mutation_label'].unique(), key=mutation_sort_key_wt_first)

    color_scale = alt.Scale(
        domain=mutations_in_plot,
        range=[('#000000' if m == 'WT Long (geomean)' else global_color_mapping.get(m, '#999999')) for m in mutations_in_plot]
    )

    legend = alt.Legend(
        labelExpr="indexof(datum.label, 'WT ') == 0 ? 'wildtype' : datum.label"
    )

    x_domain = _tight_log_domain(merged['fc_b1'])
    y_domain = _tight_log_domain(merged['fc_long'])

    x_scale = alt.Scale(type='log', domain=x_domain)
    y_scale = alt.Scale(type='log', domain=y_domain)

    points = alt.Chart(merged).mark_circle(
        size=300, opacity=0.85, stroke='black', strokeWidth=0.5
    ).encode(
        x=alt.X('fc_b1:Q', title='Fold change IC50 from WT in subtype B (B1)', scale=x_scale),
        y=alt.Y('fc_long:Q', title='Fold change IC50 from WT (Long)', scale=y_scale),
        color=alt.Color('mutation_label:N', title='Mutation', scale=color_scale, sort=mutations_in_plot, legend=legend),
        tooltip=[
            alt.Tooltip('mutation_label:N', title='Mutation'),
            alt.Tooltip('fc_b1:Q', title='B1 fold change', format='.3f'),
            alt.Tooltip('fc_long:Q', title='Long fold change', format='.3f')
        ]
    )

    layers = [points]

    min_fc = min(merged['fc_b1'].min(), merged['fc_long'].min())
    max_fc = max(merged['fc_b1'].max(), merged['fc_long'].max())
    xs = np.logspace(np.log10(min_fc), np.log10(max_fc), 200)
    id_df = pd.DataFrame({'fc_b1': xs, 'fc_long': xs})
    identity_line = alt.Chart(id_df).mark_line(
        strokeDash=[4, 4], strokeWidth=0.75, opacity=0.6, color='#808080'
    ).encode(
        x=alt.X('fc_b1:Q', scale=x_scale),
        y=alt.Y('fc_long:Q', scale=y_scale)
    )
    layers.append(identity_line)

    if merged.get('at_upper_b1', pd.Series(dtype=bool)).any():
        lod_b1 = merged.loc[merged['at_upper_b1'], 'fc_b1'].max()
        lod_df = pd.DataFrame({'fc_b1': [lod_b1]})
        lod_rule = alt.Chart(lod_df).mark_rule(
            strokeDash=[2, 3], strokeWidth=0.5, opacity=0.6, color='black'
        ).encode(
            x=alt.X('fc_b1:Q', scale=x_scale)
        )
        layers.append(lod_rule)

    if merged.get('at_upper_long', pd.Series(dtype=bool)).any():
        lod_long = merged.loc[merged['at_upper_long'], 'fc_long'].max()
        lod_df = pd.DataFrame({'fc_long': [lod_long]})
        lod_rule = alt.Chart(lod_df).mark_rule(
            strokeDash=[2, 3], strokeWidth=0.5, opacity=0.6, color='black'
        ).encode(
            y=alt.Y('fc_long:Q', scale=y_scale)
        )
        layers.append(lod_rule)

    pearson_r = pearson_p = np.nan
    if merged['fc_b1'].nunique() >= 2 and merged['fc_long'].nunique() >= 2:
        pearson_r, pearson_p = scipy_stats.pearsonr(
            np.log10(merged['fc_b1']), np.log10(merged['fc_long'])
        )

    r_label = "r = NA" if not np.isfinite(pearson_r) else f"r = {pearson_r:.2f}"
    r_df = pd.DataFrame({
        'fc_b1': [merged['fc_b1'].min()],
        'fc_long': [merged['fc_long'].max() * 0.9],
        'label': [r_label]
    })
    r_text = alt.Chart(r_df).mark_text(
        align='left', baseline='top', dx=6, dy=6, fontSize=24
    ).encode(
        x=alt.X('fc_b1:Q', scale=x_scale),
        y=alt.Y('fc_long:Q', scale=y_scale),
        text='label:N'
    )
    layers.append(r_text)

    chart = alt.layer(*layers).properties(
        width=350,
        height=280,
        title=f"{antibody} {fmt} — B1 vs Long"
    ).configure_view(
        strokeWidth=0
    ).configure_axis(
        labelFontSize=24,
        titleFontSize=26,
        labelFont='Arial',
        titleFont='Arial',
        grid=False,
        domainWidth=0.5,
        tickWidth=0.5,
        domainColor='black',
        tickColor='black',
        tickSize=5,
        tickCount=5
    ).configure_legend(
        titleFontSize=24,
        labelFontSize=22,
        labelFont='Arial',
        titleFont='Arial',
        strokeWidth=0.5,
        symbolStrokeWidth=0.5,
        symbolSize=250
    ).configure_title(
        fontSize=26,
        font='Arial'
    ).configure_text(
        font='Arial'
    )

    stats_dict = {
        'antibody': antibody,
        'format': fmt,
        'n_total': len(merged),
        'pearson_r': pearson_r,
        'pearson_p': pearson_p
    }

    return chart, stats_dict





## 1. Load Validation Strain Mutations

In [4]:
val = pd.read_excel('01_data/other/validation_mutations.xlsx')

# Keep background too (you'll want it for Long vs B1)
val = val[['Name', 'mutation', 'background']].copy()
val.columns = ['strain', 'mutation', 'strain_background']  # standard name

# Parse mutation only for non-empty strings
val['mutation'] = val['mutation'].astype(str)
val.loc[val['mutation'].str.lower().isin(['nan', 'none']), 'mutation'] = ''

val['has_mutation'] = val['mutation'].str.strip().ne('')

# Only parse where there is a mutation
val[['site', 'mutant', 'wt_residue']] = None

import re
def parse_mutation(mutation_str):
    match = re.match(r'([A-Z])(\d+)([A-Z])', str(mutation_str))
    if match:
        wildtype, site, mutant = match.groups()
        return int(site), mutant, wildtype
    return None, None, None

val.loc[val['has_mutation'], ['site', 'mutant', 'wt_residue']] = (
    val.loc[val['has_mutation'], 'mutation'].apply(lambda x: pd.Series(parse_mutation(x))).values
)

display(val.head(10))
print("WT controls retained:", (~val['has_mutation']).sum())



,strain,mutation,strain_background,has_mutation,site,mutant,wt_residue
0,RSV_B_F_PP_002W1BG.1,G446E,B_PP_002W1BG.1,True,446.0,E,G
1,RSV_A_F_PP_001WGC0,K445N,A_PP_001WGC0,True,445.0,N,K
2,RSV_A_F_PP_0046MV6.2,D73N,A_PP_0046MV6.2,True,73.0,N,D
3,RSV_A_F_PP_001QYN9,K68E,A_PP_001QYN9,True,68.0,E,K
4,RSV_B_F_PP_002WHEU,K201I,B_PP_002WHEU,True,201.0,I,K
5,RSV_B_F_PP_002SUFP,K201S,B_PP_002SUFP,True,201.0,S,K
6,RSV_A_F_PP_00463BT.1,L204S,A_PP_00463BT.1,True,204.0,S,L
7,RSV_B_F_Spain/PP_0031EHE.2,P205S,B_PP_0031EHE.2,True,205.0,S,P
8,RSV_B_F_PP_002WWH8.1,N208D,B_PP_002WWH8.1,True,208.0,D,N
9,RSV_A_F_PP_002XVQT,K209E,A_PP_002XVQT,True,209.0,E,K


WT controls retained: 2


## 2. Load IC50 Data from Neutralization Experiments

In [5]:
ic50_data = pd.read_csv('03_output/point_mut_combined_fold_changes_vs_references.csv')

# Load IC50, fold change, and other relevant columns
ic50 = ic50_data[['serum', 'virus', 'ic50', 'ic50_str', 'fc_vs_Long', 'fc_vs_B1']].copy()
ic50 = ic50.rename(columns={'virus': 'strain'})  # <-- key standardization
ic50[['antibody', 'format']] = ic50['serum'].str.rsplit(' ', n=1, expand=True)

# Sanity check: ensure Clesrovimab mutants are present in IC50 table
cles_mut_mask = (
    (ic50['antibody'] == 'Clesrovimab') &
    ic50['strain'].astype(str).str.contains(r'[A-Z]\d+[A-Z]')
)
cles_mut_count = int(cles_mut_mask.sum())
if cles_mut_count == 0:
    print("WARNING: No Clesrovimab mutants found in IC50 table. ")
    print("  - Check that combined_frac_infect.csv includes point mutants for 251212/251217")
    print("  - Re-run Point_mutant_validations-testing.ipynb to regenerate outputs")
else:
    print(f"Clesrovimab mutants in IC50 table: {cles_mut_count}")

# -------------------------------------------------------
# NEW: Parse strain_background + mutation from IC50 strain
# -------------------------------------------------------
import re

def parse_background_from_strain(s):
    s = str(s)
    # Match Long or B1 with spaces, underscores, or at word boundaries
    if re.search(r'(^|[\s_])Long([\s_]|$)', s):
        return "Long"
    if re.search(r'(^|[\s_])B1([\s_]|$)', s):
        return "B1"
    return np.nan

def parse_mut_from_strain(s):
    """
    Extract first AA mutation pattern like K209Q from strain string.
    Returns (wt_residue, site, mutant) or (None, None, None)
    """
    s = str(s)
    m = re.search(r'([A-Z])(\d+)([A-Z])', s)
    if m:
        wt, site, mut = m.groups()
        return wt, int(site), mut
    return None, None, None

# Assign background from name (works for mutants + most strains)
ic50['strain_background'] = ic50['strain'].apply(parse_background_from_strain)

# Assign mutation components from name (works for strains like ..._K209Q)
ic50[['wt_residue', 'site', 'mutant']] = ic50['strain'].apply(
    lambda x: pd.Series(parse_mut_from_strain(x))
)

ic50['has_mutation'] = ic50['site'].notna()

# Normalize merge keys to avoid dtype mismatches
ic50['antibody'] = ic50['antibody'].str.strip()
ic50['format'] = ic50['format'].str.strip()
ic50['site'] = ic50['site'].astype('Int64')
ic50['mutant'] = ic50['mutant'].astype(str)

# Assign background-appropriate fold change
ic50['fold_change'] = ic50.apply(
    lambda row: row['fc_vs_Long'] if row['strain_background'] == 'Long'
                else row['fc_vs_B1'],
    axis=1
)

print("IC50 rows with parsed background:", ic50['strain_background'].notna().sum())
print("IC50 rows with parsed mutation:", ic50['has_mutation'].sum())
print("IC50 rows with fold change:", ic50['fold_change'].notna().sum())
display(ic50.loc[ic50['has_mutation'], ['strain','strain_background','wt_residue','site','mutant','fold_change']].drop_duplicates().head(15))



Clesrovimab mutants in IC50 table: 16
IC50 rows with parsed background: 86
IC50 rows with parsed mutation: 66
IC50 rows with fold change: 86


,strain,strain_background,wt_residue,site,mutant,fold_change
0,RSV Long F K201S,Long,K,201,S,0.606946
1,RSV Long F K68Q,Long,K,68,Q,1.735751
2,RSV Long F K68N,Long,K,68,N,1.990814
3,RSV Long F K201T,Long,K,201,T,2.011009
4,RSV B1 F N201S,B1,N,201,S,14.091398
5,RSV B1 F N201T,B1,N,201,T,28.907209
6,RSV B1 F K68Q,B1,K,68,Q,32.261898
7,RSV B1 F K68N,B1,K,68,N,15.867783
8,20_RSV_F_Long_S211R,Long,S,211,R,12.692051
9,26_RSV_F_B1_Q210T,B1,Q,210,T,310.113319


## 3. Load DMS Escape Scores

In [6]:
dms = pd.read_csv('01_data/dms/all_antibodies_per_antibody_escape.csv')

dms['mutation'] = dms['wildtype'] + dms['site'].astype(str) + dms['mutant']
dms[['antibody', 'format']] = dms['antibody'].str.rsplit('-', n=1, expand=True)  # <-- name it antibody

# Keep only columns needed for merging
dms = dms[['site', 'mutant', 'wildtype', 'antibody', 'format', 'escape']].copy()
dms = dms.rename(columns={'wildtype': 'wt_residue'})

# Normalize merge keys to avoid dtype mismatches
dms['antibody'] = dms['antibody'].str.strip()
dms['format'] = dms['format'].str.strip()
dms['site'] = dms['site'].astype('Int64')
dms['mutant'] = dms['mutant'].astype(str)

display(dms.head())
print("DMS columns:", list(dms.columns))



,site,mutant,wt_residue,antibody,format,escape
0,26,A,Q,Nirsevimab,IgG,0.07960
1,26,C,Q,Nirsevimab,IgG,0.03003
2,26,D,Q,Nirsevimab,IgG,0.10040
3,26,E,Q,Nirsevimab,IgG,0.03745
4,26,F,Q,Nirsevimab,IgG,-0.36490


DMS columns: ['site', 'mutant', 'wt_residue', 'antibody', 'format', 'escape']


## 4. Merge Datasets

Merge validation mutations + IC50 data + DMS scores to create master dataframe

In [7]:
# ----------------------------
# 4. Merge Datasets (NEW)
# ----------------------------

# Start from ic50 (already has strain_background + parsed site/mutant/wt_residue)
correlation_data = ic50.copy()

# Merge DMS escape using parsed mutation + antibody/format
# NOTE: Do NOT merge on wt_residue - strain backgrounds may have different WT residues
# than the DMS reference sequence (e.g., B1 has Q at site 209, but DMS reference has K)
correlation_data = correlation_data.merge(
    dms,
    on=['site', 'mutant', 'antibody', 'format'],
    how='left',
    suffixes=('_strain', '_dms'),
    validate='many_to_one'
)

# WT controls by exact strain names
WT_LONG_STRAINS = {"40. Long GS4", "40_Long GS4"}
WT_B1_STRAINS   = {"42. B1", "42_B1"}
WT_ALL = WT_LONG_STRAINS | WT_B1_STRAINS

correlation_data['is_wt'] = correlation_data['strain'].isin(WT_ALL)

# Assign escape=0 for WT controls
correlation_data.loc[correlation_data['is_wt'], 'escape'] = 0.0

# Force their backgrounds (in case parsing didn't catch dots/spaces)
correlation_data.loc[correlation_data['strain'].isin(WT_LONG_STRAINS), 'strain_background'] = 'Long'
correlation_data.loc[correlation_data['strain'].isin(WT_B1_STRAINS),   'strain_background'] = 'B1'

print("After IC50↔DMS merge:", correlation_data.shape)
print("Rows with escape present:", correlation_data['escape'].notna().sum())
print("WT rows flagged:", correlation_data['is_wt'].sum())

# Key diagnostic: mutant rows missing escape (should be small / interpretable)
missing_mut = correlation_data.query("has_mutation == True and escape.isna()", engine="python").shape[0]
print("Mutant rows missing escape:", missing_mut)

# Show cases where strain WT differs from DMS WT (these are valid - A/B background differences)
wt_mismatch = correlation_data[
    (correlation_data['has_mutation'] == True) & 
    (correlation_data['escape'].notna()) &
    (correlation_data['wt_residue_strain'] != correlation_data['wt_residue_dms'])
]
if len(wt_mismatch) > 0:
    print(f"\nNote: {len(wt_mismatch)} mutations have different WT between strain and DMS reference:")
    print("  (This is expected for A vs B strain backgrounds)")
    for idx, row in wt_mismatch[['strain', 'wt_residue_strain', 'site', 'mutant', 'wt_residue_dms']].drop_duplicates().head(5).iterrows():
        print(f"  - {row['strain']}: strain WT={row['wt_residue_strain']}, DMS WT={row['wt_residue_dms']} at site {int(row['site'])}")

display(correlation_data.head(10))



After IC50↔DMS merge: (86, 17)
Rows with escape present: 78
WT rows flagged: 16
Mutant rows missing escape: 4

Note: 6 mutations have different WT between strain and DMS reference:
  (This is expected for A vs B strain backgrounds)
  - RSV B1 F N201S: strain WT=N, DMS WT=K at site 201
  - RSV B1 F N201T: strain WT=N, DMS WT=K at site 201
  - 28_RSV_F_B1_Q209D: strain WT=Q, DMS WT=K at site 209


,serum,strain,ic50,ic50_str,fc_vs_Long,fc_vs_B1,antibody,format,strain_background,wt_residue_strain,site,mutant,has_mutation,fold_change,wt_residue_dms,escape,is_wt
0,Nirsevimab IgG,RSV Long F K201S,0.002617,0.00262,0.606946,0.148790,Nirsevimab,IgG,Long,K,201,S,True,0.606946,K,0.31500,False
1,Nirsevimab IgG,RSV Long F K68Q,0.007483,0.00748,1.735751,0.425512,Nirsevimab,IgG,Long,K,68,Q,True,1.735751,K,-0.06815,False
2,Nirsevimab IgG,RSV Long F K68N,0.008582,0.00858,1.990814,0.488040,Nirsevimab,IgG,Long,K,68,N,True,1.990814,K,-0.18320,False
3,Nirsevimab IgG,RSV Long F K201T,0.008669,0.00867,2.011009,0.492990,Nirsevimab,IgG,Long,K,201,T,True,2.011009,K,0.24370,False
4,Nirsevimab IgG,RSV B1 F N201S,0.247804,0.248,57.481705,14.091398,Nirsevimab,IgG,B1,N,201,S,True,14.091398,K,0.31500,False
5,Nirsevimab IgG,RSV B1 F N201T,0.508346,0.508,117.918436,28.907209,Nirsevimab,IgG,B1,N,201,T,True,28.907209,K,0.24370,False
6,Nirsevimab IgG,RSV B1 F K68Q,0.567340,0.567,131.602903,32.261898,Nirsevimab,IgG,B1,K,68,Q,True,32.261898,K,-0.06815,False
7,Nirsevimab IgG,RSV B1 F K68N,0.279042,0.279,64.727941,15.867783,Nirsevimab,IgG,B1,K,68,N,True,15.867783,K,-0.18320,False
8,Nirsevimab IgG,20_RSV_F_Long_S211R,0.104525,0.105,12.692051,4.861960,Nirsevimab,IgG,Long,S,211,R,True,12.692051,S,0.65430,False
9,Nirsevimab IgG,26_RSV_F_B1_Q210T,6.667000,>6.67,809.544792,310.113319,Nirsevimab,IgG,B1,Q,210,T,True,310.113319,Q,1.31400,False


In [8]:
WT_LONG_STRAINS = {
    "RSV Long F WT ",     # 250828 (note trailing space)
    "40_Long GS4",        # 251212 (underscore)
    "40. Long GS4",       # 251217 (period)
    "RSV Long 1"          # 2025.07.24
}

WT_B1_STRAINS = {
    "RSV B1 F WT",        # 251002
    "42_B1",              # 251212 (underscore)
    "42. B1",             # 251217 (period)
    "RSV B1 1"            # 2025.07.24
}

WT_ALL = WT_LONG_STRAINS | WT_B1_STRAINS

correlation_data = correlation_data.copy()

# Flag WT
correlation_data["is_wt"] = correlation_data["strain"].isin(WT_ALL)

# Assign DMS escape = 0 for WT (so they survive escape.notna() filtering)
correlation_data.loc[correlation_data["is_wt"], "escape"] = 0.0

# Ensure they appear in the correct panels
correlation_data.loc[correlation_data["strain"].isin(WT_LONG_STRAINS), "strain_background"] = "Long"
correlation_data.loc[correlation_data["strain"].isin(WT_B1_STRAINS),   "strain_background"] = "B1"

print("WT rows flagged:", correlation_data["is_wt"].sum())
display(
    correlation_data.loc[correlation_data["is_wt"],
                         ["serum","antibody","format","strain","strain_background","escape","ic50","ic50_str"]]
    .sort_values(["serum","strain_background","strain"])
)



WT rows flagged: 20


,serum,antibody,format,strain,strain_background,escape,ic50,ic50_str
76,Clesrovimab Fab,Clesrovimab,Fab,42. B1,B1,0.0,0.072239,0.0722
75,Clesrovimab Fab,Clesrovimab,Fab,42_B1,B1,0.0,0.048574,0.0486
77,Clesrovimab Fab,Clesrovimab,Fab,40. Long GS4,Long,0.0,0.059996,0.06
74,Clesrovimab Fab,Clesrovimab,Fab,40_Long GS4,Long,0.0,0.032102,0.0321
64,Clesrovimab IgG,Clesrovimab,IgG,42. B1,B1,0.0,0.006314,0.00631
63,Clesrovimab IgG,Clesrovimab,IgG,42_B1,B1,0.0,0.003894,0.00389
65,Clesrovimab IgG,Clesrovimab,IgG,40. Long GS4,Long,0.0,0.007394,0.00739
62,Clesrovimab IgG,Clesrovimab,IgG,40_Long GS4,Long,0.0,0.003346,0.00335
58,Nirsevimab Fab,Nirsevimab,Fab,42. B1,B1,0.0,6.929534,6.93
49,Nirsevimab Fab,Nirsevimab,Fab,42_B1,B1,0.0,3.490619,3.49


In [9]:
WT_LONG_STRAINS = {
    "RSV Long F WT ",     # 250828 (note trailing space)
    "40_Long GS4",        # 251212 (underscore)
    "40. Long GS4",       # 251217 (period)
    "RSV Long 1"          # 2025.07.24
}

WT_B1_STRAINS = {
    "RSV B1 F WT",        # 251002
    "42_B1",              # 251212 (underscore)
    "42. B1",             # 251217 (period)
    "RSV B1 1"            # 2025.07.24
}

WT_ALL = WT_LONG_STRAINS | WT_B1_STRAINS

correlation_data = correlation_data.copy()

# Flag WT
correlation_data["is_wt"] = correlation_data["strain"].isin(WT_ALL)

# Assign DMS escape = 0 for WT (so they survive escape.notna() filtering)
correlation_data.loc[correlation_data["is_wt"], "escape"] = 0.0

# Ensure they appear in the correct panels
correlation_data.loc[correlation_data["strain"].isin(WT_LONG_STRAINS), "strain_background"] = "Long"
correlation_data.loc[correlation_data["strain"].isin(WT_B1_STRAINS),   "strain_background"] = "B1"

print("WT rows flagged:", correlation_data["is_wt"].sum())
display(
    correlation_data.loc[correlation_data["is_wt"],
                         ["serum","antibody","format","strain","strain_background","escape","ic50","ic50_str"]]
    .sort_values(["serum","strain_background","strain"])
)



WT rows flagged: 20


,serum,antibody,format,strain,strain_background,escape,ic50,ic50_str
76,Clesrovimab Fab,Clesrovimab,Fab,42. B1,B1,0.0,0.072239,0.0722
75,Clesrovimab Fab,Clesrovimab,Fab,42_B1,B1,0.0,0.048574,0.0486
77,Clesrovimab Fab,Clesrovimab,Fab,40. Long GS4,Long,0.0,0.059996,0.06
74,Clesrovimab Fab,Clesrovimab,Fab,40_Long GS4,Long,0.0,0.032102,0.0321
64,Clesrovimab IgG,Clesrovimab,IgG,42. B1,B1,0.0,0.006314,0.00631
63,Clesrovimab IgG,Clesrovimab,IgG,42_B1,B1,0.0,0.003894,0.00389
65,Clesrovimab IgG,Clesrovimab,IgG,40. Long GS4,Long,0.0,0.007394,0.00739
62,Clesrovimab IgG,Clesrovimab,IgG,40_Long GS4,Long,0.0,0.003346,0.00335
58,Nirsevimab Fab,Nirsevimab,Fab,42. B1,B1,0.0,6.929534,6.93
49,Nirsevimab Fab,Nirsevimab,Fab,42_B1,B1,0.0,3.490619,3.49


## 5. Filter to Data with Both IC50 and DMS Scores

For correlation analysis, we need both measurements

In [10]:
# ----------------------------
# 5. Filter to rows with IC50 + escape
# ----------------------------
# Include ALL WT controls for geometric mean calculation

correlation_data_filtered = correlation_data[
    correlation_data['escape'].notna() &
    correlation_data['ic50_str'].notna() &
    correlation_data['strain_background'].notna()
].copy()

correlation_data_filtered['at_upper_bound'] = correlation_data_filtered['ic50_str'].str.contains('>', na=False)

print(f"Filtered dataset for correlation: {len(correlation_data_filtered)} rows")
print("By background:", correlation_data_filtered['strain_background'].value_counts().to_dict())
print("WT retained after filtering:", int(correlation_data_filtered['is_wt'].sum()))
print("WT strains retained:", sorted(correlation_data_filtered[correlation_data_filtered['is_wt']]['strain'].unique()))
print("By antibody/format:")
display(correlation_data_filtered.groupby(['antibody','format','strain_background']).size())



Filtered dataset for correlation: 82 rows
By background: {'Long': 44, 'B1': 38}
WT retained after filtering: 20
WT strains retained: ['40. Long GS4', '40_Long GS4', '42. B1', '42_B1', 'RSV B1 F WT', 'RSV Long F WT ']
By antibody/format:


antibody     format  strain_background
Clesrovimab  Fab     B1                    6
                     Long                  6
             IgG     B1                    6
                     Long                  6
Nirsevimab   Fab     B1                   13
                     Long                 16
             IgG     B1                   13
                     Long                 16
dtype: int64

In [11]:
# Helper: ensure a WT row exists for a given dataset
# Uses correlation_data_filtered to compute geometric mean IC50 per antibody/format/background

def ensure_wt_row(data, strain_bg, antibody, fmt):
    if 'is_wt' in data.columns and data['is_wt'].any():
        return data

    wt_rows = correlation_data_filtered[
        (correlation_data_filtered['strain_background'] == strain_bg) &
        (correlation_data_filtered['antibody'] == antibody) &
        (correlation_data_filtered['format'] == fmt) &
        (correlation_data_filtered['is_wt'])
    ].copy()

    if len(wt_rows) == 0:
        return data

    geometric_mean_ic50 = float((wt_rows['ic50'].apply(lambda x: x) ).pipe(lambda s: (s > 0)).any() and __import__('numpy').exp(__import__('numpy').log(wt_rows['ic50']).mean()))
    wt_single = wt_rows.iloc[0].copy()
    wt_single['ic50'] = geometric_mean_ic50
    wt_single['ic50_str'] = f"{geometric_mean_ic50:.4g}"
    wt_single['fold_change'] = 1.0
    wt_single['escape'] = 0.0
    wt_single['is_wt'] = True
    wt_single['at_upper_bound'] = False
    wt_single['strain'] = f"WT {strain_bg} (geomean)"
    wt_single['site'] = None
    wt_single['mutant'] = None
    wt_single['wt_residue_strain'] = None

    data = data.copy()
    data = __import__('pandas').concat([data, __import__('pandas').DataFrame([wt_single])], ignore_index=True)
    return data



In [12]:
from IPython.display import display
import altair as alt

alt.renderers.enable("default")

# ======================
# Master color mapping function (same as neutralization curves)
# ======================
def get_master_color_mapping():
    """
    Master color mapping matching reference notebook colors for shared mutations.
    Reference: point_mutant_validations_1212and1217-Copy1.ipynb
    
    Colors preserve the exact assignments from reference notebook for all shared mutations.
    New mutations (only in this notebook) receive distinct vibrant colors.
    """
    color_mapping = {
        # From reference notebook (13 mutations) - PRESERVE EXACT COLORS
        '73N': '#1f77b4',    # blue
        '205S': '#ff7f0e',   # orange
        '207E': '#2ca02c',   # green
        '209D': '#d62728',   # red
        '209Q': '#17becf',   # aqua/cyan
        '210T': '#9467bd',   # purple
        '211R': '#8c564b',   # brown
        '215K': '#e377c2',   # pink
        '67T': '#bcbd22',    # yellow-green/olive
        '429M': '#bcbd22',   # yellow-green/olive (same as 67T)
        '446D': '#ff7f0e',   # orange (same as 205S)
        '443P': '#1f77b4',   # blue (same as 73N)
        '429S': '#2ca02c',   # green (same as 207E)
        
        # New mutations (5 mutations only in target notebooks) - ASSIGN DISTINCT COLORS
        '68N': '#FF1493',    # deep pink/magenta
        '68Q': '#4B0082',    # indigo (dark purple)
        '201S': '#008B8B',   # dark cyan/teal
        '201T': '#32CD32',   # lime green
        '67N': '#9932CC',    # dark orchid
    }
    return color_mapping

# ======================
# Create global color mapping for all mutations
# ======================
def make_mutation_label(row):
    if row.get('is_wt', False):
        # WT controls get special handling
        strain_name = str(row.get('strain', ''))
        if strain_name in ['40_Long GS4', '40. Long GS4', 'RSV Long F WT ', 'RSV Long 1']:
            return "WT Long (geomean)"  # Will be replaced by plotting function
        elif strain_name in ['42_B1', '42. B1', 'RSV B1 F WT', 'RSV B1 1']:
            return "WT B1 (geomean)"  # Will be replaced by plotting function
        else:
            return "Unmutated"
    if pd.isna(row.get('site')) or pd.isna(row.get('mutant')):
        return "Unmutated"
    site = int(row['site'])
    # Use strain WT residue if available, otherwise DMS WT
    wt = row.get('wt_residue_strain', row.get('wt_residue_dms', ''))
    return f"{wt}{site}{row.get('mutant','')}"

# Add mutation labels to the entire dataset
correlation_data_filtered['mutation_label'] = correlation_data_filtered.apply(make_mutation_label, axis=1)

# Get all unique mutations across all data
all_mutations = sorted(correlation_data_filtered['mutation_label'].unique())

# Get master color mapping (same as neutralization curves)
master_color_map = get_master_color_mapping()

# Build global_color_mapping for mutation labels
global_color_mapping = {}

for mut_label in all_mutations:
    if mut_label == "WT Long (geomean)":
        global_color_mapping[mut_label] = 'black'
    elif mut_label == "WT B1 (geomean)":
        global_color_mapping[mut_label] = '#4D4D4D'
    elif mut_label == "Unmutated":
        global_color_mapping[mut_label] = '#999999'
    else:
        # Extract position+mutant from label (e.g., "K209D" -> "209D")
        match = re.search(r'(\d+)([A-Z])$', mut_label)
        if match:
            position, mutant = match.groups()
            mutation_key = f"{position}{mutant}"
            color = master_color_map.get(mutation_key, '#999999')  # gray fallback
            global_color_mapping[mut_label] = color
        else:
            global_color_mapping[mut_label] = '#999999'  # fallback

print(f"Global color mapping created for {len(all_mutations)} unique mutations")
print("Using color scheme from reference notebook (exact color preservation)")
print("\nColor assignments:")
for mut, color in sorted(global_color_mapping.items()):
    print(f"  {mut}: {color}")
print()

# ======================
# Create individual plots with global color mapping
# ======================
individual_plots = []
individual_stats = []

strain_backgrounds = ['Long', 'B1']
antibodies = ['Nirsevimab', 'Clesrovimab']
formats = ['IgG', 'Fab']

print("Creating individual plots:")
print("=" * 80)

for strain_bg in strain_backgrounds:
    for antibody in antibodies:
        for fmt in formats:

            subset_n = len(
                correlation_data_filtered[
                    (correlation_data_filtered['strain_background'] == strain_bg) &
                    (correlation_data_filtered['antibody'] == antibody) &
                    (correlation_data_filtered['format'] == fmt)
                ]
            )

            print(f"{antibody} {fmt} — {strain_bg}: n = {subset_n}")

            chart, stats_dict = create_individual_plot(
                correlation_data_filtered,
                strain_bg,
                antibody,
                fmt,
                global_color_mapping
            )

            if chart is None:
                print("  ⚠ skipped (insufficient data)\n")
                continue

            filename = f'03_output/DMS_validation/{antibody}_{fmt}_{strain_bg}_DMS_vs_IC50.html'
            chart.save(filename)

            individual_plots.append(chart)
            individual_stats.append(stats_dict)

            display(chart)
            print()

print(f"\n✓ Displayed {len(individual_plots)} plots")



Global color mapping created for 22 unique mutations
Using color scheme from reference notebook (exact color preservation)

Color assignments:
  D73N: #1f77b4
  G446D: #ff7f0e
  K201S: #008B8B
  K201T: #32CD32
  K209D: #d62728
  K209Q: #17becf
  K68N: #FF1493
  K68Q: #4B0082
  N201S: #008B8B
  N201T: #32CD32
  N67T: #bcbd22
  P205S: #ff7f0e
  Q209D: #d62728
  Q210T: #9467bd
  R429M: #bcbd22
  R429S: #2ca02c
  S211R: #8c564b
  S215K: #e377c2
  S443P: #1f77b4
  V207E: #2ca02c
  WT B1 (geomean): #4D4D4D
  WT Long (geomean): black

Creating individual plots:
Nirsevimab IgG — Long: n = 16


/tmp/ipykernel_37836/2636672240.py:59: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  subset = pd.concat([subset, pd.DataFrame([wt_single_row])], ignore_index=True)


alt.LayerChart(...)


Nirsevimab Fab — Long: n = 16


/tmp/ipykernel_37836/2636672240.py:59: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  subset = pd.concat([subset, pd.DataFrame([wt_single_row])], ignore_index=True)


...

alt.LayerChart(...)


Clesrovimab Fab — B1: n = 6


/tmp/ipykernel_37836/2636672240.py:59: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  subset = pd.concat([subset, pd.DataFrame([wt_single_row])], ignore_index=True)


alt.LayerChart(...)



✓ Displayed 8 plots


## Fold Change vs DMS Escape Correlation

Generate correlation plots showing Fold Change IC50 vs DMS Escape scores

In [13]:
# ======================
# Generate fold change correlation plots
# ======================
from IPython.display import display

fold_change_plots = []
fold_change_stats = []

print("Creating fold change vs DMS escape plots:")
print("=" * 80)

for strain_bg in strain_backgrounds:
    for antibody in antibodies:
        for fmt in formats:

            subset_n = len(
                correlation_data_filtered[
                    (correlation_data_filtered['strain_background'] == strain_bg) &
                    (correlation_data_filtered['antibody'] == antibody) &
                    (correlation_data_filtered['format'] == fmt)
                ]
            )

            print(f"{antibody} {fmt} — {strain_bg}: n = {subset_n}")

            chart, stats_dict = create_fold_change_plot(
                correlation_data_filtered,
                strain_bg,
                antibody,
                fmt,
                global_color_mapping
            )

            if chart is None:
                print("  ⚠ skipped (insufficient data)\n")
                continue

            filename = f'03_output/DMS_validation/{antibody}_{fmt}_{strain_bg}_foldchange_vs_escape.html'
            chart.save(filename)

            fold_change_plots.append(chart)
            fold_change_stats.append(stats_dict)

            display(chart)
            print()

print(f"\n✓ Displayed {len(fold_change_plots)} fold change plots")



Creating fold change vs DMS escape plots:
Nirsevimab IgG — Long: n = 16


/tmp/ipykernel_37836/2636672240.py:237: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  subset = pd.concat([subset, pd.DataFrame([wt_single_row])], ignore_index=True)


alt.LayerChart(...)


Nirsevimab Fab — Long: n = 16


/tmp/ipykernel_37836/2636672240.py:237: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  subset = pd.concat([subset, pd.DataFrame([wt_single_row])], ignore_index=True)


...

alt.LayerChart(...)


Clesrovimab Fab — B1: n = 6


/tmp/ipykernel_37836/2636672240.py:237: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  subset = pd.concat([subset, pd.DataFrame([wt_single_row])], ignore_index=True)


alt.LayerChart(...)



✓ Displayed 8 fold change plots


## Cross-Format Correlation: Nirsevimab IgG Neutralization vs DMS Escape (B1 Background)

These plots compare **Nirsevimab IgG neutralization resistance** (fold change IC50) for B1 point mutants against **DMS escape scores** measured in both formats:

1. **IgG neutralization vs Fab DMS escape**: Does Fab binding escape predict IgG neutralization resistance?
2. **IgG neutralization vs IgG DMS escape**: Standard format-matched correlation

Both DMS escape scores are measured in the Long background (DMS library strain).

**Note**: This analysis includes ALL B1 mutants without filtering.

In [14]:
# Filter IC50 data to Nirsevimab IgG, B1 background
# Use correlation_data_filtered (which has at_upper_bound column)
ic50_nirs_igg_b1 = correlation_data_filtered[
    (correlation_data_filtered['antibody'] == 'Nirsevimab') &
    (correlation_data_filtered['format'] == 'IgG') &
    (correlation_data_filtered['strain_background'] == 'B1')
].copy()

# Merge with Fab DMS escape
data_igg_vs_fab = ic50_nirs_igg_b1[['strain', 'ic50', 'ic50_str', 'fold_change',
                                     'site', 'mutant', 'wt_residue_strain',
                                     'is_wt', 'at_upper_bound', 'antibody', 'format', 'strain_background']].copy()

fab_escape = dms[(dms['antibody'] == 'Nirsevimab') & (dms['format'] == 'Fab')][['site', 'mutant', 'escape']].copy()
data_igg_vs_fab = data_igg_vs_fab.merge(fab_escape, on=['site', 'mutant'], how='left')

# Merge with IgG DMS escape
data_igg_vs_igg = ic50_nirs_igg_b1[['strain', 'ic50', 'ic50_str', 'fold_change',
                                     'site', 'mutant', 'wt_residue_strain',
                                     'is_wt', 'at_upper_bound', 'antibody', 'format', 'strain_background']].copy()

igg_escape = dms[(dms['antibody'] == 'Nirsevimab') & (dms['format'] == 'IgG')][['site', 'mutant', 'escape']].copy()
data_igg_vs_igg = data_igg_vs_igg.merge(igg_escape, on=['site', 'mutant'], how='left')

# Set WT escape to 0 for both datasets
data_igg_vs_fab.loc[data_igg_vs_fab['is_wt'], 'escape'] = 0.0
data_igg_vs_igg.loc[data_igg_vs_igg['is_wt'], 'escape'] = 0.0

# Filter to rows with escape data (NO POSITION 201 FILTERING per user request)
data_igg_vs_fab = data_igg_vs_fab[
    data_igg_vs_fab['escape'].notna()
].copy()

data_igg_vs_igg = data_igg_vs_igg[
    data_igg_vs_igg['escape'].notna()
].copy()

# Ensure WT rows
data_igg_vs_fab = ensure_wt_row(data_igg_vs_fab, 'B1', 'Nirsevimab', 'IgG')
data_igg_vs_igg = ensure_wt_row(data_igg_vs_igg, 'B1', 'Nirsevimab', 'IgG')

print(f"IgG vs Fab escape: {len(data_igg_vs_fab)} rows")
print(f"IgG vs IgG escape: {len(data_igg_vs_igg)} rows")
print(f"Note: Includes ALL mutations (no position 201 filtering)")




IgG vs Fab escape: 13 rows
IgG vs IgG escape: 13 rows
Note: Includes ALL mutations (no position 201 filtering)


In [15]:
# Plot 1: IgG neutralization vs Fab escape
chart_fab, stats_fab = create_fold_change_plot(
    data_igg_vs_fab,
    strain_bg='B1',
    antibody='Nirsevimab',
    fmt='IgG',
    global_color_mapping=global_color_mapping
)

if chart_fab:
    # Modify title to clarify cross-format comparison
    chart_fab = chart_fab.properties(
        title='Nirsevimab IgG Neutralization (B1) vs Fab DMS Escape (Long)'
    )
    filename_fab = '03_output/DMS_validation/Nirsevimab_IgG-B1_vs_Fab-escape.html'
    chart_fab.save(filename_fab)
    display(chart_fab)
    print(f"Saved: {filename_fab}")
    print(f"Pearson r = {stats_fab['pearson_r']:.3f}, p = {stats_fab['pearson_p']:.3e}\n")

# Plot 2: IgG neutralization vs IgG escape
chart_igg, stats_igg = create_fold_change_plot(
    data_igg_vs_igg,
    strain_bg='B1',
    antibody='Nirsevimab',
    fmt='IgG',
    global_color_mapping=global_color_mapping
)

if chart_igg:
    chart_igg = chart_igg.properties(
        title='Nirsevimab IgG Neutralization (B1) vs IgG DMS Escape (Long)'
    )
    filename_igg = '03_output/DMS_validation/Nirsevimab_IgG-B1_vs_IgG-escape.html'
    chart_igg.save(filename_igg)
    display(chart_igg)
    print(f"Saved: {filename_igg}")
    print(f"Pearson r = {stats_igg['pearson_r']:.3f}, p = {stats_igg['pearson_p']:.3e}")



# Ensure WT rows
data_igg_vs_fab = ensure_wt_row(data_igg_vs_fab, 'B1', 'Nirsevimab', 'IgG')
data_igg_vs_igg = ensure_wt_row(data_igg_vs_igg, 'B1', 'Nirsevimab', 'IgG')



/tmp/ipykernel_37836/2636672240.py:237: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  subset = pd.concat([subset, pd.DataFrame([wt_single_row])], ignore_index=True)


alt.LayerChart(...)

Saved: 03_output/DMS_validation/Nirsevimab_IgG-B1_vs_Fab-escape.html
Pearson r = 0.633, p = 3.653e-02



/tmp/ipykernel_37836/2636672240.py:237: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  subset = pd.concat([subset, pd.DataFrame([wt_single_row])], ignore_index=True)


alt.LayerChart(...)

Saved: 03_output/DMS_validation/Nirsevimab_IgG-B1_vs_IgG-escape.html
Pearson r = 0.257, p = 4.459e-01


## Cross-Format Correlation: Nirsevimab Fab Neutralization vs DMS Escape (B1 Background)

These plots compare **Nirsevimab Fab neutralization resistance** (fold change IC50) for B1 point mutants against **DMS escape scores** measured in both formats:

1. **Fab neutralization vs Fab DMS escape**: Format-matched correlation
2. **Fab neutralization vs IgG DMS escape**: Does IgG binding escape predict Fab neutralization resistance?

Both DMS escape scores are measured in the Long background (DMS library strain).

In [16]:
# Filter IC50 data to Nirsevimab Fab, B1 background
ic50_nirs_fab_b1 = correlation_data_filtered[
    (correlation_data_filtered['antibody'] == 'Nirsevimab') &
    (correlation_data_filtered['format'] == 'Fab') &
    (correlation_data_filtered['strain_background'] == 'B1')
].copy()

# Merge with Fab DMS escape
data_fab_vs_fab = ic50_nirs_fab_b1[['strain', 'ic50', 'ic50_str', 'fold_change',
                                     'site', 'mutant', 'wt_residue_strain',
                                     'is_wt', 'at_upper_bound', 'antibody', 'format', 'strain_background']].copy()

fab_escape = dms[(dms['antibody'] == 'Nirsevimab') & (dms['format'] == 'Fab')][['site', 'mutant', 'escape']].copy()
data_fab_vs_fab = data_fab_vs_fab.merge(fab_escape, on=['site', 'mutant'], how='left')

# Merge with IgG DMS escape
data_fab_vs_igg = ic50_nirs_fab_b1[['strain', 'ic50', 'ic50_str', 'fold_change',
                                     'site', 'mutant', 'wt_residue_strain',
                                     'is_wt', 'at_upper_bound', 'antibody', 'format', 'strain_background']].copy()

igg_escape = dms[(dms['antibody'] == 'Nirsevimab') & (dms['format'] == 'IgG')][['site', 'mutant', 'escape']].copy()
data_fab_vs_igg = data_fab_vs_igg.merge(igg_escape, on=['site', 'mutant'], how='left')

# Set WT escape to 0 for both datasets
data_fab_vs_fab.loc[data_fab_vs_fab['is_wt'], 'escape'] = 0.0
data_fab_vs_igg.loc[data_fab_vs_igg['is_wt'], 'escape'] = 0.0

# Filter to rows with escape data
data_fab_vs_fab = data_fab_vs_fab[
    data_fab_vs_fab['escape'].notna()
].copy()

data_fab_vs_igg = data_fab_vs_igg[
    data_fab_vs_igg['escape'].notna()
].copy()

# Ensure WT rows
data_fab_vs_fab = ensure_wt_row(data_fab_vs_fab, 'B1', 'Nirsevimab', 'Fab')
data_fab_vs_igg = ensure_wt_row(data_fab_vs_igg, 'B1', 'Nirsevimab', 'Fab')

print(f"Fab vs Fab escape: {len(data_fab_vs_fab)} rows")
print(f"Fab vs IgG escape: {len(data_fab_vs_igg)} rows")




Fab vs Fab escape: 13 rows
Fab vs IgG escape: 13 rows


In [17]:
# Plot 1: Fab neutralization vs Fab escape
chart_fab_fab, stats_fab_fab = create_fold_change_plot(
    data_fab_vs_fab,
    strain_bg='B1',
    antibody='Nirsevimab',
    fmt='Fab',
    global_color_mapping=global_color_mapping
)

if chart_fab_fab:
    chart_fab_fab = chart_fab_fab.properties(
        title='Nirsevimab Fab Neutralization (B1) vs Fab DMS Escape (Long)'
    )
    filename_fab_fab = '03_output/DMS_validation/Nirsevimab_Fab-B1_vs_Fab-escape.html'
    chart_fab_fab.save(filename_fab_fab)
    display(chart_fab_fab)
    print(f"Saved: {filename_fab_fab}")
    print(f"Pearson r = {stats_fab_fab['pearson_r']:.3f}, p = {stats_fab_fab['pearson_p']:.3e}\n")

# Plot 2: Fab neutralization vs IgG escape
chart_fab_igg, stats_fab_igg = create_fold_change_plot(
    data_fab_vs_igg,
    strain_bg='B1',
    antibody='Nirsevimab',
    fmt='Fab',
    global_color_mapping=global_color_mapping
)

if chart_fab_igg:
    chart_fab_igg = chart_fab_igg.properties(
        title='Nirsevimab Fab Neutralization (B1) vs IgG DMS Escape (Long)'
    )
    filename_fab_igg = '03_output/DMS_validation/Nirsevimab_Fab-B1_vs_IgG-escape.html'
    chart_fab_igg.save(filename_fab_igg)
    display(chart_fab_igg)
    print(f"Saved: {filename_fab_igg}")
    print(f"Pearson r = {stats_fab_igg['pearson_r']:.3f}, p = {stats_fab_igg['pearson_p']:.3e}")



# Ensure WT rows
data_fab_vs_fab = ensure_wt_row(data_fab_vs_fab, 'B1', 'Nirsevimab', 'Fab')
data_fab_vs_igg = ensure_wt_row(data_fab_vs_igg, 'B1', 'Nirsevimab', 'Fab')



/tmp/ipykernel_37836/2636672240.py:237: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  subset = pd.concat([subset, pd.DataFrame([wt_single_row])], ignore_index=True)


alt.LayerChart(...)

Saved: 03_output/DMS_validation/Nirsevimab_Fab-B1_vs_Fab-escape.html
Pearson r = 0.537, p = 8.833e-02



/tmp/ipykernel_37836/2636672240.py:237: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  subset = pd.concat([subset, pd.DataFrame([wt_single_row])], ignore_index=True)


alt.LayerChart(...)

Saved: 03_output/DMS_validation/Nirsevimab_Fab-B1_vs_IgG-escape.html
Pearson r = 0.058, p = 8.663e-01


## Cross-Format Neutralization Correlation: Fab vs IgG

These plots directly compare **Fab neutralization** (fold change IC50) to **IgG neutralization** (fold change IC50) for the same mutations:

1. **Long background**: Fab vs IgG fold change for Long mutants
2. **B1 background**: Fab vs IgG fold change for B1 mutants

Both formats are measured on the same mutant viruses, allowing direct comparison of neutralization potency.

In [18]:
# ======================
# Generate Fab vs IgG neutralization correlation plots
# ======================
from IPython.display import display

format_comparison_plots = []
format_comparison_stats = []

print("Creating Fab vs IgG neutralization correlation plots:")
print("=" * 80)

for strain_bg in ['Long', 'B1']:
    print(f"\n{strain_bg} background:")

    chart, stats_dict = create_format_comparison_plot(
        correlation_data_filtered,
        strain_bg,
        antibody='Nirsevimab',
        global_color_mapping=global_color_mapping
    )

    if chart is None:
        print("  ⚠ skipped (insufficient data)\n")
        continue

    filename = f'03_output/DMS_validation/Nirsevimab_Fab-vs-IgG_{strain_bg}.html'
    chart.save(filename)

    format_comparison_plots.append(chart)
    format_comparison_stats.append(stats_dict)

    display(chart)
    print(f"Saved: {filename}")
    print(f"Pearson r = {stats_dict['pearson_r']:.3f}, p = {stats_dict['pearson_p']:.3e}")
    print(f"n = {stats_dict['n_total']} mutations\n")

print(f"\n✓ Displayed {len(format_comparison_plots)} Fab vs IgG comparison plots")



Creating Fab vs IgG neutralization correlation plots:

Long background:


/tmp/ipykernel_37836/2636672240.py:432: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  pivot_data = pd.concat([


alt.LayerChart(...)

Saved: 03_output/DMS_validation/Nirsevimab_Fab-vs-IgG_Long.html
Pearson r = 0.821, p = 3.217e-04
n = 14 mutations


B1 background:


/tmp/ipykernel_37836/2636672240.py:432: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  pivot_data = pd.concat([


alt.LayerChart(...)

Saved: 03_output/DMS_validation/Nirsevimab_Fab-vs-IgG_B1.html
Pearson r = 0.962, p = 2.162e-06
n = 11 mutations


✓ Displayed 2 Fab vs IgG comparison plots


## B1 vs Long Strain Background Comparison

Compare fold change IC50 values for Nirsevimab IgG between B1 and Long strain backgrounds.
Mutations are matched by the resulting amino acid at each site (e.g., K201S in Long vs N201S in B1).

In [19]:
# ======================
# Generate B1 vs Long comparison plot for Nirsevimab IgG
# ======================
from IPython.display import display

chart, stats = create_b1_long_comparison_plot(
    correlation_data_filtered,
    antibody='Nirsevimab',
    fmt='IgG',
    global_color_mapping=global_color_mapping
)

if chart is not None:
    display(chart)
    
    # Save the plot
    output_path = '03_output/DMS_validation/Nirsevimab_IgG_B1_vs_Long_correlation.html'
    chart.save(output_path)
    print(f"Saved: {output_path}")
else:
    print("Not enough data to create B1 vs Long comparison plot")




/tmp/ipykernel_37836/2636672240.py:643: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  merged['at_upper_b1'] = merged['at_upper_b1'].fillna(False)
/tmp/ipykernel_37836/2636672240.py:647: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  merged['at_upper_long'] = merged['at_upper_long'].fillna(False)
/tmp/ipykernel_37836/2636672240.py:661: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option

alt.LayerChart(...)

Saved: 03_output/DMS_validation/Nirsevimab_IgG_B1_vs_Long_correlation.html


In [20]:

# ======================
# Generate B1 vs Long comparison plot for Nirsevimab Fab
# ======================
from IPython.display import display

chart, stats = create_b1_long_comparison_plot(
    correlation_data_filtered,
    antibody='Nirsevimab',
    fmt='Fab',
    global_color_mapping=global_color_mapping
)

if chart is not None:
    display(chart)

    output_path = '03_output/DMS_validation/Nirsevimab_Fab_B1_vs_Long_correlation.html'
    chart.save(output_path)
    print(f'Saved: {output_path}')
else:
    print('Not enough data to create B1 vs Long comparison plot for Nirsevimab Fab')


/tmp/ipykernel_37836/2636672240.py:643: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  merged['at_upper_b1'] = merged['at_upper_b1'].fillna(False)
/tmp/ipykernel_37836/2636672240.py:647: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  merged['at_upper_long'] = merged['at_upper_long'].fillna(False)
/tmp/ipykernel_37836/2636672240.py:661: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option

alt.LayerChart(...)

Saved: 03_output/DMS_validation/Nirsevimab_Fab_B1_vs_Long_correlation.html


## Clesrovimab Cross-Format Correlations

### Cross-Format Correlation: Clesrovimab IgG Neutralization vs DMS Escape (B1 Background)

These plots compare **Clesrovimab IgG neutralization resistance** (fold change IC50) for B1 point mutants against **DMS escape scores** measured in both formats:

1. **IgG neutralization vs Fab DMS escape**: Does Fab binding escape predict IgG neutralization resistance?
2. **IgG neutralization vs IgG DMS escape**: Standard format-matched correlation

Both DMS escape scores are measured in the Long background (DMS library strain).

### Cross-Format Neutralization Correlation: Fab vs IgG (Clesrovimab)

These plots directly compare **Fab neutralization** (fold change IC50) to **IgG neutralization** (fold change IC50) for the same mutations:

1. **Long background**: Fab vs IgG fold change for Long mutants
2. **B1 background**: Fab vs IgG fold change for B1 mutants



In [21]:
# ======================
# Clesrovimab: Cross-Format Correlations (B1 background)
# ======================
from IPython.display import display

print("")
print("Clesrovimab cross-format correlations (B1 background):")
print("=" * 80)

# ----------------------
# Build Clesrovimab datasets
# ----------------------
ic50_cles_igg_b1 = correlation_data_filtered[
    (correlation_data_filtered['antibody'] == 'Clesrovimab') &
    (correlation_data_filtered['format'] == 'IgG') &
    (correlation_data_filtered['strain_background'] == 'B1')
].copy()

ic50_cles_fab_b1 = correlation_data_filtered[
    (correlation_data_filtered['antibody'] == 'Clesrovimab') &
    (correlation_data_filtered['format'] == 'Fab') &
    (correlation_data_filtered['strain_background'] == 'B1')
].copy()

# Escape datasets (Long background DMS)
cles_fab_escape = dms[(dms['antibody'] == 'Clesrovimab') & (dms['format'] == 'Fab')][['site', 'mutant', 'escape']].copy()
cles_igg_escape = dms[(dms['antibody'] == 'Clesrovimab') & (dms['format'] == 'IgG')][['site', 'mutant', 'escape']].copy()

# IgG neutralization vs Fab escape
data_cles_igg_vs_fab = ic50_cles_igg_b1[[
    'strain', 'ic50', 'ic50_str', 'fold_change',
    'site', 'mutant', 'wt_residue_strain',
    'is_wt', 'at_upper_bound', 'antibody', 'format', 'strain_background'
]].copy()

data_cles_igg_vs_fab = data_cles_igg_vs_fab.merge(cles_fab_escape, on=['site', 'mutant'], how='left')

# IgG neutralization vs IgG escape
data_cles_igg_vs_igg = ic50_cles_igg_b1[[
    'strain', 'ic50', 'ic50_str', 'fold_change',
    'site', 'mutant', 'wt_residue_strain',
    'is_wt', 'at_upper_bound', 'antibody', 'format', 'strain_background'
]].copy()

data_cles_igg_vs_igg = data_cles_igg_vs_igg.merge(cles_igg_escape, on=['site', 'mutant'], how='left')

# Fab neutralization vs Fab escape
data_cles_fab_vs_fab = ic50_cles_fab_b1[[
    'strain', 'ic50', 'ic50_str', 'fold_change',
    'site', 'mutant', 'wt_residue_strain',
    'is_wt', 'at_upper_bound', 'antibody', 'format', 'strain_background'
]].copy()

data_cles_fab_vs_fab = data_cles_fab_vs_fab.merge(cles_fab_escape, on=['site', 'mutant'], how='left')

# Fab neutralization vs IgG escape
data_cles_fab_vs_igg = ic50_cles_fab_b1[[
    'strain', 'ic50', 'ic50_str', 'fold_change',
    'site', 'mutant', 'wt_residue_strain',
    'is_wt', 'at_upper_bound', 'antibody', 'format', 'strain_background'
]].copy()

data_cles_fab_vs_igg = data_cles_fab_vs_igg.merge(cles_igg_escape, on=['site', 'mutant'], how='left')

# Set WT escape = 0
data_cles_igg_vs_fab.loc[data_cles_igg_vs_fab['is_wt'], 'escape'] = 0.0
data_cles_igg_vs_igg.loc[data_cles_igg_vs_igg['is_wt'], 'escape'] = 0.0
data_cles_fab_vs_fab.loc[data_cles_fab_vs_fab['is_wt'], 'escape'] = 0.0
data_cles_fab_vs_igg.loc[data_cles_fab_vs_igg['is_wt'], 'escape'] = 0.0

# Filter to rows with escape data
data_cles_igg_vs_fab = data_cles_igg_vs_fab[data_cles_igg_vs_fab['escape'].notna()].copy()
data_cles_igg_vs_igg = data_cles_igg_vs_igg[data_cles_igg_vs_igg['escape'].notna()].copy()
data_cles_fab_vs_fab = data_cles_fab_vs_fab[data_cles_fab_vs_fab['escape'].notna()].copy()
data_cles_fab_vs_igg = data_cles_fab_vs_igg[data_cles_fab_vs_igg['escape'].notna()].copy()

# Ensure WT rows
data_cles_igg_vs_fab = ensure_wt_row(data_cles_igg_vs_fab, 'B1', 'Clesrovimab', 'IgG')
data_cles_igg_vs_igg = ensure_wt_row(data_cles_igg_vs_igg, 'B1', 'Clesrovimab', 'IgG')
data_cles_fab_vs_fab = ensure_wt_row(data_cles_fab_vs_fab, 'B1', 'Clesrovimab', 'Fab')
data_cles_fab_vs_igg = ensure_wt_row(data_cles_fab_vs_igg, 'B1', 'Clesrovimab', 'Fab')

print(f"IgG vs Fab escape: {len(data_cles_igg_vs_fab)} rows")
print(f"IgG vs IgG escape: {len(data_cles_igg_vs_igg)} rows")
print(f"Fab vs Fab escape: {len(data_cles_fab_vs_fab)} rows")
print(f"Fab vs IgG escape: {len(data_cles_fab_vs_igg)} rows")

# Plot 1: Clesrovimab IgG neutralization vs Fab DMS escape
chart_igg_fab, stats_igg_fab = create_fold_change_plot(
    data_cles_igg_vs_fab,
    strain_bg='B1',
    antibody='Clesrovimab',
    fmt='IgG',
    global_color_mapping=global_color_mapping
)

if chart_igg_fab:
    chart_igg_fab = chart_igg_fab.properties(
        title='Clesrovimab IgG Neutralization (B1) vs Fab DMS Escape (Long)'
    )
    filename = '03_output/DMS_validation/Clesrovimab_IgG-B1_vs_Fab-escape.html'
    chart_igg_fab.save(filename)
    display(chart_igg_fab)
    print(f"Saved: {filename}")
    print(f"Pearson r = {stats_igg_fab['pearson_r']:.3f}, p = {stats_igg_fab['pearson_p']:.3e}")
else:
    print('  skipped (insufficient data for IgG vs Fab escape)')

print('')
# Plot 2: Clesrovimab IgG neutralization vs IgG DMS escape
chart_igg_igg, stats_igg_igg = create_fold_change_plot(
    data_cles_igg_vs_igg,
    strain_bg='B1',
    antibody='Clesrovimab',
    fmt='IgG',
    global_color_mapping=global_color_mapping
)

if chart_igg_igg:
    chart_igg_igg = chart_igg_igg.properties(
        title='Clesrovimab IgG Neutralization (B1) vs IgG DMS Escape (Long)'
    )
    filename = '03_output/DMS_validation/Clesrovimab_IgG-B1_vs_IgG-escape.html'
    chart_igg_igg.save(filename)
    display(chart_igg_igg)
    print(f"Saved: {filename}")
    print(f"Pearson r = {stats_igg_igg['pearson_r']:.3f}, p = {stats_igg_igg['pearson_p']:.3e}")
else:
    print('  skipped (insufficient data for IgG vs IgG escape)')

print('')
# Plot 3: Clesrovimab Fab neutralization vs Fab DMS escape
chart_fab_fab, stats_fab_fab = create_fold_change_plot(
    data_cles_fab_vs_fab,
    strain_bg='B1',
    antibody='Clesrovimab',
    fmt='Fab',
    global_color_mapping=global_color_mapping
)

if chart_fab_fab:
    chart_fab_fab = chart_fab_fab.properties(
        title='Clesrovimab Fab Neutralization (B1) vs Fab DMS Escape (Long)'
    )
    filename = '03_output/DMS_validation/Clesrovimab_Fab-B1_vs_Fab-escape.html'
    chart_fab_fab.save(filename)
    display(chart_fab_fab)
    print(f"Saved: {filename}")
    print(f"Pearson r = {stats_fab_fab['pearson_r']:.3f}, p = {stats_fab_fab['pearson_p']:.3e}")
else:
    print('  skipped (insufficient data for Fab vs Fab escape)')

print('')
# Plot 4: Clesrovimab Fab neutralization vs IgG DMS escape
chart_fab_igg, stats_fab_igg = create_fold_change_plot(
    data_cles_fab_vs_igg,
    strain_bg='B1',
    antibody='Clesrovimab',
    fmt='Fab',
    global_color_mapping=global_color_mapping
)

if chart_fab_igg:
    chart_fab_igg = chart_fab_igg.properties(
        title='Clesrovimab Fab Neutralization (B1) vs IgG DMS Escape (Long)'
    )
    filename = '03_output/DMS_validation/Clesrovimab_Fab-B1_vs_IgG-escape.html'
    chart_fab_igg.save(filename)
    display(chart_fab_igg)
    print(f"Saved: {filename}")
    print(f"Pearson r = {stats_fab_igg['pearson_r']:.3f}, p = {stats_fab_igg['pearson_p']:.3e}")
else:
    print('  skipped (insufficient data for Fab vs IgG escape)')




Clesrovimab cross-format correlations (B1 background):
IgG vs Fab escape: 6 rows
IgG vs IgG escape: 6 rows
Fab vs Fab escape: 6 rows
Fab vs IgG escape: 6 rows


/tmp/ipykernel_37836/2636672240.py:237: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  subset = pd.concat([subset, pd.DataFrame([wt_single_row])], ignore_index=True)


alt.LayerChart(...)

Saved: 03_output/DMS_validation/Clesrovimab_IgG-B1_vs_Fab-escape.html
Pearson r = 0.587, p = 2.979e-01



/tmp/ipykernel_37836/2636672240.py:237: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  subset = pd.concat([subset, pd.DataFrame([wt_single_row])], ignore_index=True)


...

alt.LayerChart(...)

Saved: 03_output/DMS_validation/Clesrovimab_Fab-B1_vs_Fab-escape.html
Pearson r = 0.795, p = 1.077e-01



/tmp/ipykernel_37836/2636672240.py:237: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  subset = pd.concat([subset, pd.DataFrame([wt_single_row])], ignore_index=True)


alt.LayerChart(...)

Saved: 03_output/DMS_validation/Clesrovimab_Fab-B1_vs_IgG-escape.html
Pearson r = 0.759, p = 1.367e-01


## Clesrovimab Fab vs IgG Neutralization Correlation

These plots directly compare **Fab neutralization** (fold change IC50) to **IgG neutralization** (fold change IC50) for the same mutations:

1. **Long background**: Fab vs IgG fold change for Long mutants
2. **B1 background**: Fab vs IgG fold change for B1 mutants



In [22]:
# ======================
# Generate Fab vs IgG neutralization correlation plots (Clesrovimab)
# ======================
from IPython.display import display

print('Creating Fab vs IgG neutralization correlation plots (Clesrovimab):')
print('=' * 80)

for strain_bg in ['Long', 'B1']:
    print('')
    print(f'{strain_bg} background:')

    chart, stats_dict = create_format_comparison_plot(
        correlation_data_filtered,
        strain_bg,
        antibody='Clesrovimab',
        global_color_mapping=global_color_mapping
    )

    if chart is None:
        print('  skipped (insufficient data)')
        print('')
        continue

    chart = chart.properties(
        title=f'Clesrovimab — {strain_bg} (IgG vs Fab)'
    )

    filename = f'03_output/DMS_validation/Clesrovimab_{strain_bg}_Fab_vs_IgG.html'
    chart.save(filename)
    display(chart)
    print(f'Saved: {filename}')
    print(f'Pearson r = {stats_dict["pearson_r"]:.3f}, p = {stats_dict["pearson_p"]:.3e}')
    print('')


Creating Fab vs IgG neutralization correlation plots (Clesrovimab):

Long background:


/tmp/ipykernel_37836/2636672240.py:432: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  pivot_data = pd.concat([


alt.LayerChart(...)

Saved: 03_output/DMS_validation/Clesrovimab_Long_Fab_vs_IgG.html
Pearson r = 0.898, p = 3.866e-02


B1 background:


/tmp/ipykernel_37836/2636672240.py:432: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  pivot_data = pd.concat([


alt.LayerChart(...)

Saved: 03_output/DMS_validation/Clesrovimab_B1_Fab_vs_IgG.html
Pearson r = 0.918, p = 2.780e-02



## Clesrovimab B1 vs Long Strain Background Comparison

Compare fold change IC50 values for Clesrovimab between B1 and Long strain backgrounds.



In [23]:
# ======================
# Generate B1 vs Long comparison plots for Clesrovimab (IgG and Fab)
# ======================
from IPython.display import display

for fmt in ['IgG', 'Fab']:
    print('')
    print(f'Clesrovimab {fmt} — B1 vs Long:')

    chart, stats = create_b1_long_comparison_plot(
        correlation_data_filtered,
        antibody='Clesrovimab',
        fmt=fmt,
        global_color_mapping=global_color_mapping
    )

    if chart is not None:
        display(chart)
        output_path = f'03_output/DMS_validation/Clesrovimab_{fmt}_B1_vs_Long_correlation.html'
        chart.save(output_path)
        print(f'Saved: {output_path}')
    else:
        print(f'Not enough data to create B1 vs Long comparison plot for Clesrovimab {fmt}')


/tmp/ipykernel_37836/2636672240.py:643: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  merged['at_upper_b1'] = merged['at_upper_b1'].fillna(False)
/tmp/ipykernel_37836/2636672240.py:647: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  merged['at_upper_long'] = merged['at_upper_long'].fillna(False)
/tmp/ipykernel_37836/2636672240.py:661: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option


Clesrovimab IgG — B1 vs Long:


alt.LayerChart(...)

Saved: 03_output/DMS_validation/Clesrovimab_IgG_B1_vs_Long_correlation.html

Clesrovimab Fab — B1 vs Long:


/tmp/ipykernel_37836/2636672240.py:643: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  merged['at_upper_b1'] = merged['at_upper_b1'].fillna(False)
/tmp/ipykernel_37836/2636672240.py:647: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  merged['at_upper_long'] = merged['at_upper_long'].fillna(False)
/tmp/ipykernel_37836/2636672240.py:661: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option

alt.LayerChart(...)

Saved: 03_output/DMS_validation/Clesrovimab_Fab_B1_vs_Long_correlation.html
